In [36]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
the_dataset = pd.read_csv("./train.csv")
the_dataset

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
Heart Disease,,,,,,,,,,,,,,
Absence,347546,347546,347546,347546,347546,347546,347546,347546,347546,347546,347546,347546,347546,347546
Presence,282454,282454,282454,282454,282454,282454,282454,282454,282454,282454,282454,282454,282454,282454


In [40]:
the_dataset.info()

the_dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Age                      630000 non-null  int64  
 1   Sex                      630000 non-null  int64  
 2   Chest pain type          630000 non-null  int64  
 3   Cholesterol              630000 non-null  int64  
 4   EKG results              630000 non-null  int64  
 5   Max HR                   630000 non-null  int64  
 6   Exercise angina          630000 non-null  int64  
 7   ST depression            630000 non-null  float64
 8   Slope of ST              630000 non-null  int64  
 9   Number of vessels fluro  630000 non-null  int64  
 10  Thallium                 630000 non-null  int64  
 11  ST_interaction           630000 non-null  float64
dtypes: float64(2), int64(10)
memory usage: 57.7 MB


,Age,Sex,Chest pain type,Cholesterol,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,ST_interaction
0,58,1,4,239,0,158,1,3.6,2,2,7,3.6
1,52,1,1,325,2,171,0,0.0,1,0,3,0.0
2,56,0,2,188,2,151,0,0.0,1,0,3,0.0
3,44,0,3,229,2,150,0,1.0,2,0,3,0.0
4,58,1,4,234,2,125,1,3.8,2,3,3,3.8
...,...,...,...,...,...,...,...,...,...,...,...,...
629995,56,0,1,226,0,132,0,0.0,1,0,7,0.0
629996,54,1,4,249,2,150,0,0.0,2,0,3,0.0
629997,67,1,4,275,0,149,0,0.0,1,2,7,0.0
629998,52,1,4,199,2,157,0,0.0,1,0,6,0.0


In [21]:
the_dataset.rename(columns={"Heart Disease_Presence": "Heart Disease"}, inplace=True)
the_dataset["ST_interaction"] = the_dataset["ST depression"] * the_dataset["Exercise angina"]

In [22]:
correlations = the_dataset.corr()["Heart Disease"]
correlations.sort_values(ascending=False)

Heart Disease              1.000000
Thallium                   0.605776
Chest pain type            0.460684
Exercise angina            0.441864
Number of vessels fluro    0.438604
ST depression              0.430641
Slope of ST                0.415050
ST_interaction             0.394162
Sex                        0.342446
EKG results                0.218961
Age                        0.212091
Cholesterol                0.082753
FBS over 120               0.033570
id                         0.000209
BP                        -0.005181
Max HR                    -0.440985
Name: Heart Disease, dtype: float64

In [29]:
### Model creation ###


heart_disease = the_dataset["Heart Disease"]
the_dataset.drop(columns=["Heart Disease"], inplace=True)

In [34]:
X_train, X_test, y_train, y_test = train_test_split(the_dataset, heart_disease, test_size=0.2, random_state=42, stratify=heart_disease)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
import numpy as np

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns



preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

# Models
models = XGBClassifier()
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
base_spw = neg / pos

param_grid = {
    "model__n_estimators": [100, 300, 500],
    "model__max_depth": [3, 4, 6, 8],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 5, 10],
    "model__gamma": [0, 0.1, 0.3],
    "model__scale_pos_weight": [
        1,
        base_spw * 0.5,
        base_spw,
        base_spw * 1.5
    ]
}

pipeline = Pipeline(steps=[
    ("scaling", preprocessor),
    ("model", models)
])

strat = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions= param_grid,
    scoring="roc_auc",
    n_iter=50,
    n_jobs=-1,
    cv=strat,
    random_state=42,
    return_train_score=True
)

random_search.fit(X_train, y_train)
print(random_search.best_estimator_)
print(random_search.best_params_)
print(random_search.best_score_)
